# Hệ thống Phân loại Trái cây tại Quầy Thu Ngân

**Đồ án CS231 - Nhập môn Thị giác Máy tính**

4 mô hình: **KNN, SVM, CNN, ResNet18**.


## Bước 1: Cài đặt thư viện

In [1]:
!pip install -q gradio scikit-image

## Bước 2: Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = '/content/drive/MyDrive/CS231.Q21/models'

import os
print('Các file trong MODEL_DIR:')
for f in os.listdir(MODEL_DIR):
    size_mb = os.path.getsize(os.path.join(MODEL_DIR, f)) / 1024 / 1024
    print(f'  {f} ({size_mb:.1f} MB)')

Mounted at /content/drive
Các file trong MODEL_DIR:
  SVM_scaler_hog_color.joblib (0.0 MB)
  SVM_hog_color_model.joblib (45.7 MB)
  SVM_scaler_hog_only.joblib (0.0 MB)
  ResNet18_best.pth (42.7 MB)
  CNN_best_model.keras (1.3 MB)
  CNN_final_model.keras (1.3 MB)
  KNN_best.pkl (102.5 MB)
  KNN_metadata.json (0.0 MB)
  KNN_pca.pkl (653.3 MB)
  KNN_scaler.pkl (0.6 MB)


## Bước 3: Import và cấu hình

In [3]:
import os
import joblib
import numpy as np
import cv2
from PIL import Image

import gradio as gr

import tensorflow as tf
from tensorflow import keras

import torch
import torch.nn as nn
from torchvision import models, transforms

from skimage.feature import hog
from sklearn.pipeline import Pipeline

CLASS_NAMES = ['apple', 'banana', 'mango', 'orange', 'tomato']
CLASS_NAMES_VI = {
    'apple': 'Táo',
    'banana': 'Chuối',
    'mango': 'Xoài',
    'orange': 'Cam',
    'tomato': 'Cà chua',
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## Bước 4: Hàm trích xuất đặc trưng

In [ ]:
def extract_hog_for_knn(gray_image):
    return hog(
        gray_image,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        feature_vector=True,
    )


def extract_color_histogram_for_knn(image_uint8):
    hsv = cv2.cvtColor(image_uint8, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [16, 8, 4],
                        [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()


def extract_hog_for_svm(gray_image):
    return hog(
        gray_image,
        orientations=8,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        feature_vector=True,
    )


def extract_color_histogram_for_svm(image_uint8):
    # Theo code gốc user cung cấp: [8, 8, 8] bins = 512 features
    # HOG (1568) + Color (512) = 2080 features
    hsv = cv2.cvtColor(image_uint8, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8],
                        [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

## Bước 5: Hàm tiền xử lý cho từng mô hình

In [ ]:
def preprocess_for_knn(pil_image):
    img = np.array(pil_image.convert('RGB').resize((224, 224)))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    hog_feat = extract_hog_for_knn(gray)
    color_feat = extract_color_histogram_for_knn(img)
    return np.concatenate([hog_feat, color_feat]).reshape(1, -1)


def preprocess_for_svm(pil_image):
    img = np.array(pil_image.convert('RGB').resize((128, 128)))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    hog_feat = extract_hog_for_svm(gray)
    color_feat = extract_color_histogram_for_svm(img)
    return np.concatenate([hog_feat, color_feat]).reshape(1, -1)


def preprocess_for_cnn(pil_image):
    img = np.array(pil_image.convert('RGB').resize((224, 224)))
    return np.expand_dims(img, axis=0).astype(np.float32)


resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


def preprocess_for_resnet(pil_image):
    return resnet_transform(pil_image.convert('RGB')).unsqueeze(0).to(device)

## Bước 6: Load các mô hình

**Lưu ý về SVM:**
- File `svm_model.joblib` có thể là một `Pipeline` (đã tích hợp StandardScaler bên trong) hoặc một `SVC` thuần.
- Code sẽ tự động kiểm tra và in ra loại để bạn biết.

In [ ]:
print('Đang load các mô hình...\n')

# --- KNN ---
knn_model = joblib.load(os.path.join(MODEL_DIR, 'knn_best.pkl'))
knn_scaler = joblib.load(os.path.join(MODEL_DIR, 'knn_scaler.pkl'))
knn_pca = joblib.load(os.path.join(MODEL_DIR, 'knn_pca.pkl'))
print('  ✓ KNN loaded')

# --- SVM ---
svm_model = joblib.load(os.path.join(MODEL_DIR, 'svm_hog_color_model.joblib'))

# Thử load scaler với tên file cụ thể của người dùng hoặc các tên mặc định
svm_scaler = None
possible_scaler_names = ['scaler_hog_color.joblib']

for name in possible_scaler_names:
    path = os.path.join(MODEL_DIR, name)
    if os.path.exists(path):
        svm_scaler = joblib.load(path)
        print(f'  ✓ SVM Scaler loaded ({name})')
        break

if isinstance(svm_model, Pipeline):
    print(f'  ✓ SVM loaded (Pipeline với các bước: {list(svm_model.named_steps.keys())})')
else:
    print(f'  ✓ SVM loaded (model thuần: {type(svm_model).__name__})')
    if svm_scaler is None:
        print('    ⚠️  Cảnh báo: Không tìm thấy scaler cho SVM. SVM có thể dự đoán sai.')

# --- CNN ---
cnn_model = keras.models.load_model(os.path.join(MODEL_DIR, 'best_model.keras'))
print('  ✓ CNN loaded')

# --- ResNet18 ---
resnet_model = models.resnet18(weights=None)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 5)
resnet_model.load_state_dict(
    torch.load(os.path.join(MODEL_DIR, 'resnet18_best.pth'), map_location=device)
)
resnet_model = resnet_model.to(device).eval()
print('  ✓ ResNet18 loaded')

print('\nTất cả mô hình đã sẵn sàng.')

Đang load các mô hình...

  ✓ KNN loaded
  ✓ SVM Scaler loaded (scaler_hog_color.joblib)
  ✓ SVM loaded (model thuần: SVC)
  ✓ CNN loaded
  ✓ ResNet18 loaded

Tất cả mô hình đã sẵn sàng.


In [ ]:
if isinstance(svm_model, Pipeline):
    actual_svc = svm_model.steps[-1][1]
else:
    actual_svc = svm_model

if hasattr(actual_svc, 'probability'):
    print(f'Mô hình SVM đang bật chế độ probability: {actual_svc.probability}')
    if not actual_svc.probability:
        print('=> Hiện tại SVM đang dùng hàm softmax thủ công trên decision_function để tính xác suất.')
else:
    print('Mô hình không phải là SVC của sklearn hoặc không có thuộc tính probability.')

Mô hình SVM đang bật chế độ probability: False
=> Hiện tại SVM đang dùng hàm softmax thủ công trên decision_function để tính xác suất.


## Bước 7: Hàm dự đoán

`predict_svm` xử lý cả 2 trường hợp:
- Pipeline: gọi trực tiếp `predict_proba` (Pipeline tự xử lý scaler)
- SVC thuần: dùng đặc trưng thô

In [ ]:
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()


def get_probs(model, feat):
    can_predict_proba = hasattr(model, 'predict_proba')
    if isinstance(model, Pipeline):
        last_step = model.steps[-1][1]
        if hasattr(last_step, 'probability'):
            can_predict_proba = can_predict_proba and last_step.probability
    elif hasattr(model, 'probability'):
        can_predict_proba = can_predict_proba and model.probability

    if can_predict_proba:
        return model.predict_proba(feat)[0]
    else:
        scores = model.decision_function(feat)[0]
        return softmax(scores)


def format_top3(probs):
    top3_idx = np.argsort(probs)[::-1][:3]
    return {
        f'{CLASS_NAMES_VI[CLASS_NAMES[i]]} ({CLASS_NAMES[i]})': float(probs[i])
        for i in top3_idx
    }


def run_knn(pil_image):
    if pil_image is None: return {}
    probs = predict_knn(pil_image)
    return format_top3(probs)


def run_svm(pil_image):
    if pil_image is None: return {}
    probs = predict_svm(pil_image)
    return format_top3(probs)


def run_cnn(pil_image):
    if pil_image is None: return {}
    probs = predict_cnn(pil_image)
    return format_top3(probs)


def run_resnet(pil_image):
    if pil_image is None: return {}
    probs = predict_resnet(pil_image)
    return format_top3(probs)


def predict_knn(pil_image):
    feat = preprocess_for_knn(pil_image)
    feat = knn_scaler.transform(feat)
    feat = knn_pca.transform(feat)
    return knn_model.predict_proba(feat)[0]


def predict_svm(pil_image):
    feat = preprocess_for_svm(pil_image)
    if not isinstance(svm_model, Pipeline) and svm_scaler is not None:
        feat = svm_scaler.transform(feat)
    return get_probs(svm_model, feat)


def predict_cnn(pil_image):
    x = preprocess_for_cnn(pil_image)
    return cnn_model.predict(x, verbose=0)[0]


def predict_resnet(pil_image):
    x = preprocess_for_resnet(pil_image)
    with torch.no_grad():
        logits = resnet_model(x)
        return torch.softmax(logits, dim=1)[0].cpu().numpy()

## Bước 9: Khởi chạy giao diện Gradio

Sau khi cell này chạy xong, sẽ có 2 link:
- **Local URL**: chỉ dùng được trong Colab
- **Public URL** (`https://xxxxx.gradio.live`): chia sẻ được, hết hạn sau 72h

In [ ]:
with gr.Blocks(title='Phân loại Trái cây - CS231') as demo:
    gr.Markdown("# Hệ thống Phân loại Trái cây")

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type='pil', label='Ảnh đầu vào', height=350)
            predict_btn = gr.Button('🔍 Phân loại', variant='primary')

        with gr.Column(scale=2):
            with gr.Row():
                knn_output = gr.Label(num_top_classes=3, label='KNN')
                svm_output = gr.Label(num_top_classes=3, label='SVM')
            with gr.Row():
                cnn_output = gr.Label(num_top_classes=3, label='CNN')
                resnet_output = gr.Label(num_top_classes=3, label='ResNet18')

    predict_btn.click(fn=run_knn, inputs=input_image, outputs=knn_output)
    predict_btn.click(fn=run_svm, inputs=input_image, outputs=svm_output)
    predict_btn.click(fn=run_cnn, inputs=input_image, outputs=cnn_output)
    predict_btn.click(fn=run_resnet, inputs=input_image, outputs=resnet_output)

    input_image.change(fn=run_knn, inputs=input_image, outputs=knn_output)
    input_image.change(fn=run_svm, inputs=input_image, outputs=svm_output)
    input_image.change(fn=run_cnn, inputs=input_image, outputs=cnn_output)
    input_image.change(fn=run_resnet, inputs=input_image, outputs=resnet_output)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://993fa33a86e860b410.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
